In [4]:
from scipy.stats import mannwhitneyu
import itertools
import numpy as np
from collections import defaultdict
from typing import Callable, Optional
import csv
import pandas as pd
import os
import utils

import utils


def is_valid_data_file(file_name:str) -> bool:
    return file_name.endswith("json") or file_name.endswith("txt")


def get_mean_for_combinations(df: pd.DataFrame, 
                       independent_variables: list[str], 
                       dependent_variables: list[str]) -> pd.DataFrame:

    # ensure all the columns are present in the df
    for col in independent_variables+dependent_variables:
        if col not in df:
            raise Exception(f"The column {col} is not in the dataframe\n\t(columns are {list(df.columns)})")
    assert(all(col in df for col in independent_variables))
    assert(dependent_variable in df for dependent_variable in dependent_variables)
    
    grouped = df.groupby(independent_variables, dropna=False)[dependent_variables].mean().reset_index()
    
    return grouped


import json
import os


def json_to_entries(data: dict):
    def item_to_list_of_entries(item) -> list[dict]:
        problem_name = item["problem_name"]
        pRef_method = item["pRef_method"]
        pRef_size = item["sample_size"]

        entries = item["results_by_tree"]

        def get_modified_entry(entry):
            entry["problem"] = problem_name
            entry["pRef_method"] = pRef_method
            entry["pRef_size"] = pRef_size

            errors = entry["results"]
            entry = entry | errors
            del entry["results"]

            if "order_tree" in entry:
                del entry["order_tree"]

            return entry

        entries = list(map(get_modified_entry, entries))
        return entries

    return [entry for item in data for entry in item_to_list_of_entries(item)]

def convert_accuracy_data_to_df(input_directory, output_filename):

    all_dicts = []
    # Iterate through all files in the input directory
    for filename in os.listdir(input_directory):
        # Construct full file path
        file_path = os.path.join(input_directory, filename)

        # Check if the file is a JSON file
        if not os.path.isfile(file_path):
            continue

        if not is_valid_data_file(file_path):
            continue

        with open(file_path, 'r') as file:
            data = json.load(file)
            entries = json_to_entries(data)
            all_dicts.extend(entries)

    # Convert list of dictionaries to DataFrame
    df = pd.DataFrame(all_dicts)

    # Write the DataFrame to a CSV file
    df.to_csv(output_filename, index=False)
    
    

def json_to_tree_data(data: dict):
    def item_to_list_of_entries(item) -> list[dict]:
        surrounding_information = {prop: item[prop]
                                   for prop in ["problem_name", "pRef_method"]}
        surrounding_information = {"problem": item["problem_name"],
                                   "pRef_method": item["pRef_method"]}

        entries = item["results_by_tree"]
        entries = [thing for thing in entries if "order_tree" in thing]  

        def convert_order_tree(order_tree, accumulator = None, current_depth: int = 0):
            if accumulator is None:
                accumulator = defaultdict(list)
            accumulator[current_depth].append(order_tree["own"])
            if len(order_tree["matching"]) > 0:
                convert_order_tree(order_tree["matching"], accumulator, current_depth+1)

            if len(order_tree["unmatching"]) > 0:
                convert_order_tree(order_tree["unmatching"], accumulator, current_depth+1)

            return accumulator
        def convert_tree_to_averages_by_level(entry):
            ps_search_info = {prop: entry[prop]
                                   for prop in ["ps_budget", "ps_population", "metrics"]}
            tree_structure = entry["order_tree"]
            just_depths = convert_order_tree(tree_structure)
            #average_orders_by_depth = {f"average_at_{depth}": np.average(orders)
            #                  for depth, orders in just_depths.items()}
            #standard_deviations = {f"sd_at_{depth}": np.std(orders)
            #                  for depth, orders in just_depths.items()}
            #overall_average = {"overall_average": np.average(list(itertools.chain(*(just_depths.values()))))}
            core_info_trees = [{"depth": depth,
                               "order": order}
                               for depth in just_depths
                                for order in just_depths[depth]
                               ]
            core_info_trees = [surrounding_information | ps_search_info | core_tree
            for core_tree in core_info_trees]
            return core_info_trees


        entries = list(map(convert_tree_to_averages_by_level, entries))
        return entries

    return [entry for item in data for entry in item_to_list_of_entries(item)]

def convert_tree_data_to_df(input_directory, output_filename):

    all_dicts = []
    # Iterate through all files in the input directory
    for filename in os.listdir(input_directory):
        # Construct full file path
        file_path = os.path.join(input_directory, filename)

        # Check if the file is a JSON file
        if not os.path.isfile(file_path):
            continue

        if not is_valid_data_file(file_path):
            continue


        with open(file_path, 'r') as file:
            data = json.load(file)
            entries = json_to_tree_data(data)
            all_dicts.extend(entries)

    # Convert list of dictionaries to DataFrame
    df = pd.DataFrame(all_dicts)

    # Write the DataFrame to a CSV file
    df.to_csv(output_filename, index=False)
    
    
def filter_dataframe(df, **kwargs):
    df = df.copy()  # Make a copy of the DataFrame to avoid modifying the original
    for col, value in kwargs.items():
        if col in df.columns:
            df = df[df[col] == value]
        else:
            raise ValueError(f"Column '{col}' not found in dataframe.")
    return df
        

    
    

def prettify_kind_column(df):
    df['kind'] = df.apply(
    lambda row: (
        'PS-W' if row['kind'] == 'ps' and row['metrics'] == 'variance' else
        'PS-WA' if row['kind'] == 'ps' else
        'Trad.' if row['kind'] == 'naive' else
        'IAI' if row['kind'] == 'iai' else
        row['kind']
    ),
    axis=1
)
    
    

    
    

In [3]:
#run_location = r"/Users/gian/Desktop/CondorResults/VDT/compareown/run3/"
run_location = r"C:\Users\gac8\Desktop\CondorResults\VDT\compareown\all_final_runs"

results_csv = os.path.join(run_location, "results.csv")
tree_data_csv = os.path.join(run_location, "tree_data.csv")


#convert_accuracy_data_to_df(os.path.join(run_location, "data"), results_csv)
#convert_tree_data_to_df(os.path.join(run_location, "data"), tree_data_csv)


In [4]:

accuracy_data = pd.read_csv(results_csv)
tree_data = pd.read_csv(tree_data_csv)

display(accuracy_data)
display(accuracy_data.dtypes)
display(tree_data)

prettify_kind_column(accuracy_data)
#prettify_kind_column(tree_data)


for kind in accuracy_data["kind"].unique():
    matching_rows = accuracy_data[accuracy_data['kind'] == kind]
    print(f"For the tree kind {kind}, there are {matching_rows.shape[0]} rows")

#headers = "kind,depth,ps_budget,ps_population,avoid_ancestors,metrics,problem,pRef_method,mse,mae,r_sq,evs"

,kind,depth,problem,pRef_method,pRef_size,mse,mae,r_sq,evs,ps_budget,ps_population,avoid_ancestors,metrics,cp
0,naive,2,SAT_S,uniform,10000,10.428119,2.562856,0.244222,0.244534,NaN,NaN,NaN,NaN,NaN
1,naive,3,SAT_S,uniform,10000,8.912910,2.392481,0.354036,0.354210,NaN,NaN,NaN,NaN,NaN
2,naive,4,SAT_S,uniform,10000,7.868440,2.238714,0.429734,0.430817,NaN,NaN,NaN,NaN,NaN
3,naive,5,SAT_S,uniform,10000,6.944226,2.096012,0.496717,0.497513,NaN,NaN,NaN,NaN,NaN
4,naive,6,SAT_S,uniform,10000,6.355751,2.011702,0.539367,0.539568,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94315,iai,2,BT,Tabu,10000,29.001422,4.231988,0.134516,0.134791,NaN,NaN,NaN,NaN,0.25
94316,iai,3,BT,Tabu,10000,25.743493,3.994729,0.231742,0.231858,NaN,NaN,NaN,NaN,0.25
94317,iai,4,BT,Tabu,10000,23.159979,3.810264,0.308841,0.308907,NaN,NaN,NaN,NaN,0.25
94318,iai,5,BT,Tabu,10000,20.122597,3.606257,0.399485,0.399681,NaN,NaN,NaN,NaN,0.25


kind                object
depth                int64
problem             object
pRef_method         object
pRef_size            int64
mse                float64
mae                float64
r_sq               float64
evs                float64
ps_budget          float64
ps_population      float64
avoid_ancestors     object
metrics             object
cp                 float64
dtype: object

,problem,pRef_method,ps_budget,ps_population,metrics,average_at_0,average_at_1,average_at_2,average_at_3,average_at_4,average_at_5,overall_average,sd_at_0,sd_at_1,sd_at_2,sd_at_3,sd_at_4,sd_at_5
0,SAT_S,uniform,5000,100,variance,1.0,1.5,2.25,3.250,4.125000,5.103448,4.216667,0.0,0.5,0.829156,1.561249,1.690969,2.202582
1,SAT_S,uniform,5000,100,variance estimated_atomicity,1.0,1.5,1.75,1.750,1.875000,1.937500,1.857143,0.0,0.5,0.433013,0.433013,0.330719,0.242061
2,SAT_S,uniform,5000,100,variance,2.0,1.5,2.25,3.750,3.562500,3.516129,3.387097,0.0,0.5,0.433013,1.639360,1.498697,1.965361
3,SAT_S,uniform,5000,100,variance estimated_atomicity,2.0,1.0,1.50,2.000,1.812500,1.937500,1.857143,0.0,0.0,0.500000,0.000000,0.390312,0.242061
4,SAT_S,GA,5000,100,variance,4.0,4.5,5.00,5.875,5.357143,5.840000,5.574074,0.0,2.5,1.224745,2.570870,2.438007,2.781079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10747,BT,SA,5000,100,variance estimated_atomicity,2.0,2.0,2.00,1.875,2.875000,2.206897,2.316667,0.0,0.0,0.000000,0.599479,3.314268,1.423432
10748,BT,Tabu,5000,100,variance,8.0,14.0,12.50,12.250,9.133333,8.000000,9.528302,0.0,5.0,0.866025,6.437197,4.814792,6.460583
10749,BT,Tabu,5000,100,variance estimated_atomicity,2.0,2.0,2.00,1.875,1.928571,2.681818,2.254902,0.0,0.0,0.000000,0.330719,0.593330,1.394351
10750,BT,Tabu,5000,100,variance,13.0,5.5,7.00,15.500,11.375000,9.320000,10.553571,0.0,4.5,3.605551,7.245688,6.489174,4.913003


For the tree kind Trad., there are 26880 rows
For the tree kind PS-W, there are 26880 rows
For the tree kind PS-WA, there are 26880 rows
For the tree kind IAI, there are 13680 rows


In [17]:
    
    
def generate_statistical_test_data(accuracy_data: pd.DataFrame, input_directory, output_filename):
    depths = [3, 4, 5]
    usable_data = filter_dataframe(accuracy_data, pRef_size = 10000)
    usable_data = usable_data[usable_data["depth"].isin(depths)]
    
    result_column = "r_sq"
    
    def winning_competitor_for_competition_and_values(problem: str, depth: int, metaheuristic: str) -> (str, np.ndarray):
        for_each_method = {tree_method: filter_dataframe(usable_data, problem = problem, depth = depth, pRef_method = metaheuristic, kind = tree_method)[result_column]
                           for tree_method in {"PS-W", "PS-WA", "IAI", "Trad."}}
        
        iai_average = np.average(for_each_method["IAI"])
        naive_average = np.average(for_each_method["Trad."])
        
        
        winning_competitor_method = "IAI" if iai_average > naive_average else "Trad."
        
        p_value_between_w_and_competitor = mannwhitneyu(for_each_method["PS-W"], for_each_method[winning_competitor_method], alternative="greater").pvalue
        p_value_between_wa_and_competitor = mannwhitneyu(for_each_method["PS-WA"], for_each_method[winning_competitor_method], alternative="greater").pvalue
        
        return {"problem": problem,
                "depth": depth,
                "metaheuristic": metaheuristic,
                "p_value_w":p_value_between_w_and_competitor,
                "p_value_wa": p_value_between_wa_and_competitor,
                "winning_competitor": winning_competitor_method}
    
    
    
    all_problems = usable_data["problem"].unique()
    all_metaheuristics = usable_data["pRef_method"].unique()
    
    dicts = [winning_competitor_for_competition_and_values(problem=problem, depth = depth, metaheuristic=metaheuristic)
             for problem in all_problems
             for depth in depths
             for metaheuristic in all_metaheuristics]
    
    return pd.DataFrame(dicts)
        
        
    

statistical_data = generate_statistical_test_data(accuracy_data, None, None)

pivot_table = statistical_data.pivot_table(index=["problem", "depth", "metaheuristic"], 
                                            values =["p_value_w", "p_value_wa"])
display(statistical_data)
display(pivot_table)

,problem,depth,metaheuristic,p_value_w,p_value_wa,winning_competitor
0,SAT_S,3,uniform,9.992276e-01,1.0,Trad.
1,SAT_S,3,GA,6.875231e-19,1.0,Trad.
2,SAT_S,3,SA,3.387702e-14,1.0,Trad.
3,SAT_S,3,Tabu,2.347066e-33,1.0,Trad.
4,SAT_S,4,uniform,1.000000e+00,1.0,Trad.
...,...,...,...,...,...,...
67,BT,4,Tabu,1.051677e-01,1.0,Trad.
68,BT,5,uniform,1.000000e+00,1.0,Trad.
69,BT,5,GA,7.839605e-36,1.0,Trad.
70,BT,5,SA,1.000000e+00,1.0,Trad.


p_value_w  p_value_wa
problem depth metaheuristic                          
BT      3     GA             1.800464e-38         1.0
              SA             2.912382e-12         1.0
              Tabu           4.534832e-03         1.0
              uniform        1.000000e+00         1.0
        4     GA             1.475329e-37         1.0
...                                   ...         ...
SAT_S   4     uniform        1.000000e+00         1.0
        5     GA             1.177666e-15         1.0
              SA             1.095645e-02         1.0
              Tabu           3.406477e-33         1.0
              uniform        1.000000e+00         1.0

[72 rows x 2 columns]

In [37]:
def bold_max(row):
    row_as_numbers = [float(item[:-1]) for item in row]
    max_number = max(row_as_numbers)
    
    return ['font-weight: bold' if item == max_number else '' for item in row_as_numbers]

    
def style_pivot_table(pivot_table):
    custom_column_order = ['PS-W', 'PS-WA', 'IAI', 'Trad.']

    pivot_table = pivot_table.mul(100).round(1).astype(str) + "%"    

    # Reorder columns based on custom order
    pivot_table = pivot_table.reindex(columns=custom_column_order)

    styled_df = pivot_table.style.apply(bold_max, axis=1)

    return styled_df

def put_latex_tables_side_by_side(left_latex, right_latex):
    return r"\begin{tabular}{ccccccc}\hline"+left_latex+r"\\ \hline\end{tabular}\quad\begin{tabular}{ccccccc}\hline"+right_latex+r"\\ \hline\end{tabular}"

def fix_latex(input_string):
    # Replace '%' with '\%'
    replacements = {"%":"\\%",
                    "pRef_method":"Met.",
                    "{SA}": r"{\rotcell{SA}}", # note that SA is a subset of SAT\_50 etc.., so it causes some issues
                    "SAT_S": "SAT\_20",
                    "SAT_M": "SAT\_50",
                    "SAT_L": "SAT\_100",
                    "GC_L": "GC\_anna",
                    "GC_S": "GC\_jean",
                    "uniform": "RS",
                    "kind": "tree",
                    r"\multirow[c]{12}" : r"\hline \multirow[c]{12}",
                    r"& \multirow[c]{3}" : r"\cline{2-7} & \multirow[c]{3}",
                    r"} \cline{2-7}" : "} ",
                   "\\font-weightbold": "",
                    "≪": "\ll "}

    texts_to_rotate = ["problem", "BT", "GC\_anna", "GC\_jean", "SAT\_20",  "SAT\_50",  "SAT\_100", "Met.", "GA", "Tabu", "RS", "depth"]

    for item_to_rotate in texts_to_rotate:
        replacements[item_to_rotate] = r"\rotcell{"+item_to_rotate+"}"

    modified_string = str(input_string)
    for orig, replacement in replacements.items():
        modified_string = modified_string.replace(orig, replacement)
    
    return modified_string


def pivot_table_as_latex(pivot_table):
    latex_text = pivot_table.to_latex(convert_css=True)
    latex_text = fix_latex(latex_text)
    return latex_text

In [28]:
pRef_size = 10000
depths = [3, 4, 5]


usable_data = accuracy_data.copy()
usable_data = usable_data[usable_data["pRef_size"] == pRef_size] 
usable_data = usable_data[usable_data["depth"].isin(depths)]

independent_variables = ["problem", "pRef_method", "kind", "depth"]
dependent_variables = ["r_sq"]


problems = ["BT", "GC_S", "GC_L", "SAT_S", "SAT_M", "SAT_L"]

left_problems, right_problems = problems[:3], problems[3:]

def make_table_for_problems(problem_subset):
    with_right_problems = usable_data[usable_data['problem'].isin(problem_subset)]
    pivot_table = with_right_problems.pivot_table(index = ["problem", "pRef_method", "depth"], 
                                        columns = ["kind"], 
                                        values = dependent_variables[0])
    return style_pivot_table(pivot_table)



left_table = make_table_for_problems(left_problems)
right_table = make_table_for_problems(right_problems)

display(left_table)
display(right_table)

left_table_latex = pivot_table_as_latex(left_table)
right_table_latex = pivot_table_as_latex(right_table)

full_table_latex = put_latex_tables_side_by_side(left_table_latex, right_table_latex)

print("left table:")
print(left_table_latex)

print("\n\n\n\n\n\nright table")
print(right_table_latex)



# pivot_table = usable_data.pivot_table(index = ["problem", "pRef_method", "depth"], 
#                                         columns = ["kind"], 
#                                         values = dependent_variables[0])





# for problem in usable_data['problem'].unique():
#     with_right_problem = usable_data[usable_data['problem'] == problem]
#     pivot_table = with_right_problem.pivot_table(index = ["pRef_method", "depth"], 
#                                         columns = ["kind"], 
#                                         values = dependent_variables[0])

#     pivot_table = style_pivot_table(pivot_table)
#     print(fix_latex(f"{problem = }"))
#     print_pivot_table_as_latex(pivot_table)
    
    # Display the styled dataframe
    #display(styled_df)
    
    #display(pivot_table)

left table:
\begin{tabular}{lllllll}
 &  & tree & PS-W & PS-WA & IAI & Trad. \\
\rotcell{problem} & \rotcell{Met.} & \rotcell{depth} &  &  &  &  \\
\hline \multirow[c]{12}{*}{\rotcell{BT}}  & \multirow[c]{3}{*}{\rotcell{GA}} & 3 & \bfseries 88.5\% & 72.2\% & 66.6\% & 77.3\% \\
 &  & 4 & \bfseries 90.9\% & 78.7\% & 78.0\% & 83.4\% \\
 &  & 5 & \bfseries 92.3\% & 82.9\% & 83.8\% & 86.9\% \\
 \cline{2-7} & \multirow[c]{3}{*}{\rotcell{SA}} & 3 & \bfseries 83.8\% & 62.0\% & 65.0\% & 79.2\% \\
 &  & 4 & 87.7\% & 73.8\% & 82.1\% & \bfseries 88.0\% \\
 &  & 5 & 90.4\% & 81.0\% & 89.8\% & \bfseries 92.8\% \\
 \cline{2-7} & \multirow[c]{3}{*}{\rotcell{Tabu}} & 3 & \bfseries 32.3\% & 14.6\% & 21.9\% & 30.3\% \\
 &  & 4 & \bfseries 40.1\% & 21.3\% & 32.2\% & 39.3\% \\
 &  & 5 & 47.1\% & 28.5\% & 41.8\% & \bfseries 48.8\% \\
 \cline{2-7} & \multirow[c]{3}{*}{\rotcell{RS}} & 3 & 9.9\% & 4.5\% & 11.2\% & \bfseries 14.5\% \\
 &  & 4 & 11.4\% & 5.6\% & 14.3\% & \bfseries 17.2\% \\
 &  & 5 & 12.5\% & 6.

In [45]:

pRef_size = 10000
depths = [3, 4, 5]


problems = ["BT", "GC_S", "GC_L", "SAT_S", "SAT_M", "SAT_L"]

left_problems, middle_problems, right_problems = problems[:2], problems[2:4], problems[4:]

def make_table_for_problems(problem_subset):
    with_right_problems = statistical_data[statistical_data['problem'].isin(problem_subset)]
    threshold = 0.005
    # Add the new 'successfull' column with "<<0.05" or the original p-value
    with_right_problems['p-value*'] = with_right_problems['p_value_w'].apply(lambda x: r'$\ll \alpha$' if x < threshold else str(round(x, 2))[:4])
    
    display(with_right_problems)

    pivot_table = with_right_problems.pivot_table(index = ["problem", "metaheuristic"], 
                                                  columns = ["depth"],
                                        values = ["p-value*"],
                                                  aggfunc=lambda x: x)
    
    
    return pivot_table



table = make_table_for_problems(problems)

display(table)

latex_text = table.to_latex()
latex_text = fix_latex(latex_text)
print(latex_text)

#left_table_latex = pivot_table_as_latex(left_table)
#right_table_latex = pivot_table_as_latex(right_table)

#full_table_latex = put_latex_tables_side_by_side(left_table_latex, right_table_latex)

# print("left table:")
# print(left_table_latex)
# 
# print("\n\n\n\n\n\nright table")
# print(right_table_latex)


,problem,depth,metaheuristic,p_value_w,p_value_wa,winning_competitor,p-value*
0,SAT_S,3,uniform,9.992276e-01,1.0,Trad.,1.0
1,SAT_S,3,GA,6.875231e-19,1.0,Trad.,$\ll \alpha$
2,SAT_S,3,SA,3.387702e-14,1.0,Trad.,$\ll \alpha$
3,SAT_S,3,Tabu,2.347066e-33,1.0,Trad.,$\ll \alpha$
4,SAT_S,4,uniform,1.000000e+00,1.0,Trad.,1.0
...,...,...,...,...,...,...,...
67,BT,4,Tabu,1.051677e-01,1.0,Trad.,0.11
68,BT,5,uniform,1.000000e+00,1.0,Trad.,1.0
69,BT,5,GA,7.839605e-36,1.0,Trad.,$\ll \alpha$
70,BT,5,SA,1.000000e+00,1.0,Trad.,1.0


p-value*                            
depth                             3             4             5
problem metaheuristic                                          
BT      GA             $\ll \alpha$  $\ll \alpha$  $\ll \alpha$
        SA             $\ll \alpha$          0.64           1.0
        Tabu           $\ll \alpha$          0.11          0.94
        uniform                 1.0           1.0           1.0
GC_L    GA             $\ll \alpha$  $\ll \alpha$  $\ll \alpha$
        SA             $\ll \alpha$           1.0           1.0
        Tabu                    1.0           1.0           1.0
        uniform                 1.0           1.0           1.0
GC_S    GA             $\ll \alpha$  $\ll \alpha$  $\ll \alpha$
        SA                     0.11           1.0           1.0
        Tabu                    1.0           1.0           1.0
        uniform                 1.0           1.0           1.0
SAT_L   GA             $\ll \alpha$  $\ll \alpha$  $\ll \alpha$
        SA             $\ll \alpha$  $\ll \alpha$           1.0
        Tabu           $\ll \alpha$  $\ll \alpha$          0.96
        uniform                 1.0           1.0           1.0
SAT_M   GA             $\ll \alpha$  $\ll \alpha$  $\ll \alpha$
        SA             $\ll \alpha$  $\ll \alpha$  $\ll \alpha$
        Tabu           $\ll \alpha$  $\ll \alpha$  $\ll \alpha$
        uniform                0.83           1.0           1.0
SAT_S   GA             $\ll \alpha$  $\ll \alpha$  $\ll \alpha$
        SA             $\ll \alpha$  $\ll \alpha$          0.01
        Tabu           $\ll \alpha$  $\ll \alpha$  $\ll \alpha$
        uniform                 1.0           1.0           1.0

\begin{tabular}{lllll}
\toprule
 &  & \multicolumn{3}{r}{p-value*} \\
 & \rotcell{depth} & 3 & 4 & 5 \\
\rotcell{problem} & metaheuristic &  &  &  \\
\midrule
\multirow[t]{4}{*}{\rotcell{BT}} & \rotcell{GA} & $\ll \alpha$ & $\ll \alpha$ & $\ll \alpha$ \\
 & SA & $\ll \alpha$ & 0.64 & 1.0 \\
 & \rotcell{Tabu} & $\ll \alpha$ & 0.11 & 0.94 \\
 & \rotcell{RS} & 1.0 & 1.0 & 1.0 \\
\cline{1-5}
\multirow[t]{4}{*}{\rotcell{GC\_anna}} & \rotcell{GA} & $\ll \alpha$ & $\ll \alpha$ & $\ll \alpha$ \\
 & SA & $\ll \alpha$ & 1.0 & 1.0 \\
 & \rotcell{Tabu} & 1.0 & 1.0 & 1.0 \\
 & \rotcell{RS} & 1.0 & 1.0 & 1.0 \\
\cline{1-5}
\multirow[t]{4}{*}{\rotcell{GC\_jean}} & \rotcell{GA} & $\ll \alpha$ & $\ll \alpha$ & $\ll \alpha$ \\
 & SA & 0.11 & 1.0 & 1.0 \\
 & \rotcell{Tabu} & 1.0 & 1.0 & 1.0 \\
 & \rotcell{RS} & 1.0 & 1.0 & 1.0 \\
\cline{1-5}
\multirow[t]{4}{*}{\rotcell{SAT\_100}} & \rotcell{GA} & $\ll \alpha$ & $\ll \alpha$ & $\ll \alpha$ \\
 & SA & $\ll \alpha$ & $\ll \alpha$ & 1.0 \\
 & \rotcell{Tabu} 

In [1]:

import pandas as pd

tree_data_path = r"/Users/gian/PycharmProjects/PS-descriptors/resources/variance_tree_materials/tree_data.csv"


In [2]:
tree_data = pd.read_csv(tree_data_path)
display(tree_data)

,problem,pRef_method,ps_budget,ps_population,metrics,average_at_0,average_at_1,average_at_2,average_at_3,average_at_4,average_at_5,overall_average,sd_at_0,sd_at_1,sd_at_2,sd_at_3,sd_at_4,sd_at_5
0,SAT_S,uniform,5000,100,variance,1.0,1.5,2.25,3.250,4.125000,5.103448,4.216667,0.0,0.5,0.829156,1.561249,1.690969,2.202582
1,SAT_S,uniform,5000,100,variance estimated_atomicity,1.0,1.5,1.75,1.750,1.875000,1.937500,1.857143,0.0,0.5,0.433013,0.433013,0.330719,0.242061
2,SAT_S,uniform,5000,100,variance,2.0,1.5,2.25,3.750,3.562500,3.516129,3.387097,0.0,0.5,0.433013,1.639360,1.498697,1.965361
3,SAT_S,uniform,5000,100,variance estimated_atomicity,2.0,1.0,1.50,2.000,1.812500,1.937500,1.857143,0.0,0.0,0.500000,0.000000,0.390312,0.242061
4,SAT_S,GA,5000,100,variance,4.0,4.5,5.00,5.875,5.357143,5.840000,5.574074,0.0,2.5,1.224745,2.570870,2.438007,2.781079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10747,BT,SA,5000,100,variance estimated_atomicity,2.0,2.0,2.00,1.875,2.875000,2.206897,2.316667,0.0,0.0,0.000000,0.599479,3.314268,1.423432
10748,BT,Tabu,5000,100,variance,8.0,14.0,12.50,12.250,9.133333,8.000000,9.528302,0.0,5.0,0.866025,6.437197,4.814792,6.460583
10749,BT,Tabu,5000,100,variance estimated_atomicity,2.0,2.0,2.00,1.875,1.928571,2.681818,2.254902,0.0,0.0,0.000000,0.330719,0.593330,1.394351
10750,BT,Tabu,5000,100,variance,13.0,5.5,7.00,15.500,11.375000,9.320000,10.553571,0.0,4.5,3.605551,7.245688,6.489174,4.913003


In [10]:
dependent_variables = ['average_at_0', 'sd_at_0', 'average_at_1', 'sd_at_1', 'average_at_2', 'sd_at_2', 'average_at_3', 'sd_at_3', 'average_at_4', 'sd_at_4', 'average_at_5', 'sd_at_5', 'overall_average']

independent_variables = ["metrics", "problem"]

aggregated_table = get_mean_for_combinations(filter_dataframe(tree_data, pRef_method = "GA"), 
                                             independent_variables=independent_variables,
                                             dependent_variables=dependent_variables)

display(aggregated_table)

pivot_table = aggregated_table.pivot_table(index = independent_variables,
                                           values= dependent_variables)

display(pivot_table)

,metrics,problem,average_at_0,sd_at_0,average_at_1,sd_at_1,average_at_2,sd_at_2,average_at_3,sd_at_3,average_at_4,sd_at_4,average_at_5,sd_at_5,overall_average
0,variance,BT,13.611607,0.0,19.287946,4.502232,26.147321,11.243856,28.355469,13.668180,26.713499,14.792931,23.598081,14.316307,24.949370
1,variance,GC_L,4.116071,0.0,7.194196,1.823661,12.920759,7.300021,17.047991,8.879721,18.560073,9.575224,18.524239,9.802332,17.312544
2,variance,GC_S,4.883929,0.0,7.723214,1.843750,14.135045,6.321064,21.218750,12.034704,24.006923,13.745071,24.014482,13.756657,22.109680
3,variance,SAT_L,5.785714,0.0,9.178571,2.013393,17.997768,10.158462,24.885603,14.154799,26.701751,15.024421,24.670184,13.988038,23.935699
4,variance,SAT_M,5.236607,0.0,8.598214,2.459821,11.597098,6.152232,14.359933,7.020416,15.026501,7.234208,13.941949,7.188041,13.807841
5,variance,SAT_S,2.812500,0.0,4.008929,1.071429,4.856027,2.019792,5.362564,2.270884,5.684430,2.555885,5.680336,2.619170,5.453692
6,variance estimated_atomicity,BT,2.165179,0.0,2.040179,0.218750,2.006696,0.468156,1.986522,0.584506,2.018111,0.801434,2.044472,1.072488,2.027426
7,variance estimated_atomicity,GC_L,2.187500,0.0,2.095982,0.265625,2.079241,0.420089,1.990753,0.553768,1.980214,0.623603,1.950155,0.601004,1.983524
8,variance estimated_atomicity,GC_S,2.200893,0.0,2.087054,0.212054,2.140625,0.508922,2.071189,0.658335,2.066780,0.765643,2.056786,0.842780,2.071958
9,variance estimated_atomicity,SAT_L,2.433036,0.0,2.238839,0.238839,2.223214,0.431794,2.152663,0.577835,2.089629,0.674304,2.052553,0.755305,2.103545


average_at_0  average_at_1  \
metrics                      problem                               
variance                     BT          13.611607     19.287946   
                             GC_L         4.116071      7.194196   
                             GC_S         4.883929      7.723214   
                             SAT_L        5.785714      9.178571   
                             SAT_M        5.236607      8.598214   
                             SAT_S        2.812500      4.008929   
variance estimated_atomicity BT           2.165179      2.040179   
                             GC_L         2.187500      2.095982   
                             GC_S         2.200893      2.087054   
                             SAT_L        2.433036      2.238839   
                             SAT_M        2.245536      2.113839   
                             SAT_S        1.933036      1.897321   

                                      average_at_2  average_at_3  \
metrics                      problem                               
variance                     BT          26.147321     28.355469   
                             GC_L        12.920759     17.047991   
                             GC_S        14.135045     21.218750   
                             SAT_L       17.997768     24.885603   
                             SAT_M       11.597098     14.359933   
                             SAT_S        4.856027      5.362564   
variance estimated_atomicity BT           2.006696      1.986522   
                             GC_L         2.079241      1.990753   
                             GC_S         2.140625      2.071189   
                             SAT_L        2.223214      2.152663   
                             SAT_M        2.090402      2.053731   
                             SAT_S        1.892857      1.876674   

                                      average_at_4  average_at_5  \
metrics                      problem                               
variance                     BT          26.713499     23.598081   
                             GC_L        18.560073     18.524239   
                             GC_S        24.006923     24.014482   
                             SAT_L       26.701751     24.670184   
                             SAT_M       15.026501     13.941949   
                             SAT_S        5.684430      5.680336   
variance estimated_atomicity BT           2.018111      2.044472   
                             GC_L         1.980214      1.950155   
                             GC_S         2.066780      2.056786   
                             SAT_L        2.089629      2.052553   
                             SAT_M        1.998679      1.956471   
                             SAT_S        1.874263      1.858087   

                                      overall_average  sd_at_0   sd_at_1  \
metrics                      problem                                       
variance                     BT             24.949370      0.0  4.502232   
                             GC_L           17.312544      0.0  1.823661   
                             GC_S           22.109680      0.0  1.843750   
                             SAT_L          23.935699      0.0  2.013393   
                             SAT_M          13.807841      0.0  2.459821   
                             SAT_S           5.453692      0.0  1.071429   
variance estimated_atomicity BT              2.027426      0.0  0.218750   
                             GC_L            1.983524      0.0  0.265625   
                             GC_S            2.071958      0.0  0.212054   
                             SAT_L           2.103545      0.0  0.238839   
                             SAT_M           2.001298      0.0  0.194196   
                             SAT_S           1.869814      0.0  0.089286   

                                        sd_at_2    sd_at_3    sd_at_4  \
metrics                      problem     